# Paso 2: Obtención y Carga del Conjunto de Datos

El dataset utilizado es **Online Learning Engagement Dataset**, disponible públicamente en [Kaggle](https://www.kaggle.com/).

Se descarga mediante la **API pública de Kaggle**, que es una API REST oficial que permite acceder a datasets, competiciones y kernels de forma programática. Esto cumple con el requisito de obtener los datos a través de una API pública.

### Configuración previa de la API de Kaggle
1. Crear una cuenta en [kaggle.com](https://www.kaggle.com)
2. Ir a **Account → API → Create New API Token** → descarga `kaggle.json`
3. Colocar el archivo en `~/.kaggle/kaggle.json` (Mac/Linux) o `C:\Users\<user>\.kaggle\kaggle.json` (Windows)
4. Ejecutar: `pip install kaggle`

In [ ]:
## 2.1 Instalación de dependencias necesarias
# !pip install kaggle pandas sqlalchemy

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Rutas
REPO_ROOT = Path("..").resolve()
DATA_PATH = REPO_ROOT / "online_learning_engagement_dataset.csv"
DB_PATH   = REPO_ROOT / "online_learning.db"

## 2.2 Descarga del dataset mediante la API de Kaggle

La API de Kaggle permite descargar datasets con el comando `kaggle datasets download -d <owner>/<dataset>`.  
El dataset utilizado es: **`rabieelkharoua/students-performance-dataset`**  
*(Si el archivo ya existe localmente, se omite la descarga para no repetir el proceso.)*

In [ ]:
if not DATA_PATH.exists():
    import subprocess, zipfile

    # Dataset slug de Kaggle (owner/dataset-name)
    KAGGLE_DATASET = "rabieelkharoua/students-performance-dataset"

    print("Descargando dataset desde la API de Kaggle...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
         "-p", str(REPO_ROOT), "--unzip"],
        check=True
    )
    print(f"Dataset descargado en: {DATA_PATH}")
else:
    print(f"Dataset ya disponible en: {DATA_PATH}")

## 2.3 Carga del CSV con Pandas

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas: {list(df.columns)}")
df.head()

In [ ]:
df.info()
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Paso 3: Almacenamiento en Base de Datos

Se utiliza **SQLite** como motor de base de datos, gestionado a través de **SQLAlchemy** (ORM estudiado en el curso).  

### ¿Por qué SQLite?
- No requiere servidor: el archivo `.db` vive dentro del repositorio.
- Soporta SQL estándar completo: SELECT, JOIN, INSERT, UPDATE, DELETE...
- Totalmente compatible con SQLAlchemy y `pandas.read_sql`.
- Mayor seguridad y persistencia que un CSV plano.

El dataset se divide en **dos tablas relacionadas** para poder practicar JOINs:
- `students`: datos demográficos y de acceso.
- `performance`: métricas de rendimiento académico y engagement.

In [ ]:
from sqlalchemy import create_engine, text

# Crear engine SQLite (crea el archivo .db si no existe)
engine = create_engine(f"sqlite:///{DB_PATH}")

# --- Tabla 1: students (perfil demográfico y acceso) ---
students = df[["student_id", "age", "gender", "country",
               "device_type", "internet_speed_mbps",
               "study_hours_weekly", "login_frequency_weekly"]].copy()

# --- Tabla 2: performance (métricas académicas y de engagement) ---
performance = df[["student_id", "avg_session_duration_min",
                  "video_watch_time_min", "assignments_submitted",
                  "forum_posts", "quiz_attempts", "avg_quiz_score",
                  "attendance_rate", "engagement_score",
                  "final_grade", "dropout"]].copy()

# Guardar en SQLite (reemplaza si existe)
students.to_sql("students", engine, if_exists="replace", index=False)
performance.to_sql("performance", engine, if_exists="replace", index=False)

print("Tablas creadas correctamente en SQLite:")
with engine.connect() as conn:
    tablas = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table'")).fetchall()
    for t in tablas:
        print(f"  - {t[0]}")

## 3.1 Consultas SQL — SELECT

Consultas básicas para explorar los datos y obtener información de valor previo al EDA.

In [ ]:
# SELECT: Distribución de estudiantes por país (top 10)
query_paises = """
SELECT country,
       COUNT(*) AS total_students,
       ROUND(AVG(study_hours_weekly), 2) AS avg_study_hours
FROM students
GROUP BY country
ORDER BY total_students DESC
LIMIT 10
"""
df_paises = pd.read_sql(query_paises, engine)
print("Top 10 países por número de estudiantes:")
df_paises

In [ ]:
# SELECT: Tasa de abandono (dropout) por dispositivo
query_dropout = """
SELECT device_type,
       COUNT(*) AS total,
       SUM(p.dropout) AS dropouts,
       ROUND(100.0 * SUM(p.dropout) / COUNT(*), 2) AS dropout_rate_pct
FROM students s
JOIN performance p ON s.student_id = p.student_id
GROUP BY device_type
ORDER BY dropout_rate_pct DESC
"""
df_dropout = pd.read_sql(query_dropout, engine)
print("Tasa de abandono por tipo de dispositivo:")
df_dropout

## 3.2 Consultas SQL — JOIN

Combinamos las tablas `students` y `performance` para obtener una visión completa del perfil del estudiante y su rendimiento.

In [ ]:
# JOIN: Perfil completo de estudiantes con alto riesgo de abandono
# (engagement_score bajo y attendance_rate < 0.5)
query_riesgo = """
SELECT s.student_id,
       s.age,
       s.gender,
       s.country,
       s.study_hours_weekly,
       p.engagement_score,
       p.attendance_rate,
       p.avg_quiz_score,
       p.final_grade,
       p.dropout
FROM students s
INNER JOIN performance p ON s.student_id = p.student_id
WHERE p.attendance_rate < 0.5
  AND p.engagement_score < 5
ORDER BY p.engagement_score ASC
LIMIT 15
"""
df_riesgo = pd.read_sql(query_riesgo, engine)
print(f"Estudiantes con alto riesgo de abandono: {len(df_riesgo)} registros (muestra de 15)")
df_riesgo

In [ ]:
# JOIN: Rendimiento promedio por género y velocidad de internet (alta vs baja)
query_internet = """
SELECT s.gender,
       CASE WHEN s.internet_speed_mbps >= 50 THEN 'Alta (>=50 Mbps)'
            ELSE 'Baja (<50 Mbps)' END AS internet_tier,
       COUNT(*) AS total_students,
       ROUND(AVG(p.final_grade), 2)      AS avg_final_grade,
       ROUND(AVG(p.avg_quiz_score), 2)   AS avg_quiz_score,
       ROUND(AVG(p.engagement_score), 2) AS avg_engagement
FROM students s
JOIN performance p ON s.student_id = p.student_id
GROUP BY s.gender, internet_tier
ORDER BY s.gender, internet_tier
"""
df_internet = pd.read_sql(query_internet, engine)
print("Rendimiento según género y velocidad de conexión:")
df_internet

## 3.3 Consultas SQL — INSERT

Se inserta un nuevo registro de estudiante ficticio para demostrar la escritura en la base de datos.

In [ ]:
with engine.connect() as conn:
    # INSERT en tabla students
    conn.execute(text("""
        INSERT INTO students
            (student_id, age, gender, country, device_type,
             internet_speed_mbps, study_hours_weekly, login_frequency_weekly)
        VALUES
            (999999, 28, 'Male', 'Spain', 'Laptop', 100.0, 20.0, 7)
    """))

    # INSERT en tabla performance
    conn.execute(text("""
        INSERT INTO performance
            (student_id, avg_session_duration_min, video_watch_time_min,
             assignments_submitted, forum_posts, quiz_attempts,
             avg_quiz_score, attendance_rate, engagement_score,
             final_grade, dropout)
        VALUES
            (999999, 60.0, 500.0, 10, 5, 8, 85.0, 0.95, 9.5, 88.0, 0)
    """))

    conn.commit()
    print("INSERT completado. Verificando registro insertado:")

# Verificar el INSERT con SELECT
df_nuevo = pd.read_sql(
    "SELECT s.*, p.final_grade, p.dropout FROM students s JOIN performance p ON s.student_id = p.student_id WHERE s.student_id = 999999",
    engine
)
df_nuevo

## 3.4 Consultas analíticas avanzadas (valor previo al EDA estadístico)

In [ ]:
# Segmentación de estudiantes por nivel de engagement
query_segmentos = """
SELECT
    CASE
        WHEN p.engagement_score >= 8 THEN 'Alto (>=8)'
        WHEN p.engagement_score >= 5 THEN 'Medio (5-8)'
        ELSE 'Bajo (<5)'
    END AS engagement_level,
    COUNT(*)                              AS total,
    ROUND(AVG(p.final_grade), 2)          AS avg_grade,
    ROUND(AVG(p.attendance_rate) * 100, 2) AS avg_attendance_pct,
    ROUND(AVG(s.study_hours_weekly), 2)   AS avg_study_hours,
    SUM(p.dropout)                        AS total_dropouts,
    ROUND(100.0 * SUM(p.dropout) / COUNT(*), 2) AS dropout_pct
FROM performance p
JOIN students s ON p.student_id = s.student_id
WHERE s.student_id != 999999   -- excluir registro ficticio
GROUP BY engagement_level
ORDER BY avg_grade DESC
"""
df_segmentos = pd.read_sql(query_segmentos, engine)
print("Segmentación por nivel de engagement:")
df_segmentos

In [ ]:
# Top 5 países con mayor nota final promedio (mín. 100 estudiantes)
query_top_paises = """
SELECT s.country,
       COUNT(*)                       AS total_students,
       ROUND(AVG(p.final_grade), 2)   AS avg_final_grade,
       ROUND(AVG(p.avg_quiz_score),2) AS avg_quiz_score,
       ROUND(100.0 * SUM(p.dropout) / COUNT(*), 2) AS dropout_pct
FROM students s
JOIN performance p ON s.student_id = p.student_id
WHERE s.student_id != 999999
GROUP BY s.country
HAVING COUNT(*) >= 100
ORDER BY avg_final_grade DESC
LIMIT 5
"""
df_top = pd.read_sql(query_top_paises, engine)
print("Top 5 países con mayor nota final promedio (mín. 100 estudiantes):")
df_top

## Base de datos completa

In [ ]:
df_full = pd.read_sql("""
    SELECT s.student_id, s.age, s.gender, s.country, s.device_type,
           s.internet_speed_mbps, s.study_hours_weekly, s.login_frequency_weekly,
           p.avg_session_duration_min, p.video_watch_time_min,
           p.assignments_submitted, p.forum_posts, p.quiz_attempts,
           p.avg_quiz_score, p.attendance_rate, p.engagement_score,
           p.final_grade, p.dropout
    FROM students s
    JOIN performance p ON s.student_id = p.student_id
    ORDER BY s.student_id
""", engine)

print(f"Total de registros en la base de datos: {len(df_full):,}")
df_full